# 39 · 10 classes, seed 42 · PC-CNN trained with centered EP

- experiments_in_seeds/pc_cnn_ep/39_resizemix50_seed42 (PCCNN, centered EP: K = 10, β = 0.01), with NUM_CLASSES = 10.
- Compare with cnn/39_resizemix50_seed42 (BP, same data, seed and settings): mean val epochs 96–100 = 79.52.


In [ ]:
from pathlib import Path
import torch

from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode

from src.dataset.balanced_class_subset import BalancedClassSubset
from src.dataset.tiny_imagenet_dataset import TinyImageNetLoader, get_selected_classes, get_selected_class_names
from src.utils.utils import means, stds, convert_to_rgb, get_device, print_device_info, set_seed, make_generator, count_parameters
from src.utils.plot_graph import show_mix_samples, plot_training_curves
from src.dataset.resizemix import ResizeMixDataset
from src.model.pc_cnn_ep import PCCNN
from src.utils.metrics_csv import print_metrics
from src.utils.train_model import TrainModel
from src.utils.checkpoint import CheckpointManager
from src.utils.timing import compare_step_time

RESUME = False
SEED = 42
NUM_CLASSES = 10
TRAIN_PER_CLASS = 500
VAL_PER_CLASS = 50
BATCH_SIZE = 64
NUM_WORKERS = 4
EPOCHS = 100
INITIAL_LR = 0.05
MIN_LR = 1e-05
WEIGHT_DECAY = 0.0001
LABEL_SMOOTHING = 0.1
CLASSIFIER_DROPOUT = 0.35
CLEAN_EVAL_INTERVAL = 5

RESIZEMIX_SCALE = (0.1, 0.8)
RESIZEMIX_PROB = 0.50

EP_STEPS = 10  # sweeps per nudged phase (K = 7 fails the gradient check)
EP_BETA = 0.01

TINY_ROOT = Path('../../../data/tiny_imagenet/tiny-imagenet-200')
SAVE_DIR = Path('../checkpoints/pc_cnn_ep/39_resizemix50_seed42_ep')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = get_device()

set_seed(SEED)

print('Device:', DEVICE)
print('Dataset root:', TINY_ROOT)
print('DataLoader workers:', NUM_WORKERS)
print('Epochs:', EPOCHS)
print(f'ResizeMix on {RESIZEMIX_PROB:.0%} of images | scale={RESIZEMIX_SCALE} | RandAugment after mixing')
print(f'Training: Centered EP | K={EP_STEPS} sweeps | beta={EP_BETA}')

In [ ]:
print_device_info(DEVICE)

## 1. Dataset and transforms

In [ ]:
before_mix_tf = transforms.Compose(
    [transforms.Lambda(convert_to_rgb), transforms.RandomCrop(64, padding=4, padding_mode='reflect'),
     transforms.RandomHorizontalFlip(p=0.5)])
after_mix_tf = transforms.Compose(
    [transforms.RandAugment(num_ops=2, magnitude=5, interpolation=InterpolationMode.BILINEAR), transforms.ToTensor(),
     transforms.Normalize(mean=means, std=stds)])
test_tf = transforms.Compose(
    [transforms.Lambda(convert_to_rgb), transforms.ToTensor(), transforms.Normalize(mean=means, std=stds)])

train_dataset, val_dataset, _, _ = TinyImageNetLoader(seed=SEED, root=TINY_ROOT, train_transform=before_mix_tf,
                                                      test_transform=test_tf).load()
clean_train_dataset, _, _, _ = TinyImageNetLoader(seed=SEED, root=TINY_ROOT, train_transform=test_tf,
                                                  test_transform=test_tf).load()

train_subset = BalancedClassSubset(train_dataset, get_selected_classes(NUM_CLASSES), TRAIN_PER_CLASS, seed=SEED)
clean_train_subset = BalancedClassSubset(clean_train_dataset, get_selected_classes(NUM_CLASSES), TRAIN_PER_CLASS, seed=SEED)
val_subset = BalancedClassSubset(val_dataset, get_selected_classes(NUM_CLASSES), VAL_PER_CLASS, seed=SEED + 1)

assert train_subset.samples == clean_train_subset.samples, 'Clean and augmented training subsets differ.'
assert len(train_subset) == NUM_CLASSES * TRAIN_PER_CLASS
assert len(val_subset) == NUM_CLASSES * VAL_PER_CLASS

CLASS_NAMES = get_selected_class_names(train_dataset, TINY_ROOT, NUM_CLASSES)

print('Train:', len(train_subset), 'Validation:', len(val_subset))

## 2. DataLoaders (macOS / Jupyter safe)

In [ ]:
loader_generator = make_generator(SEED)

loader_options = {'batch_size': BATCH_SIZE, 'num_workers': NUM_WORKERS, 'pin_memory': DEVICE.type == 'cuda',
                  'persistent_workers': NUM_WORKERS > 0, **({'prefetch_factor': 2} if NUM_WORKERS > 0 else {})}

train_mix_dataset = ResizeMixDataset(train_subset, after_mix_tf, scale=RESIZEMIX_SCALE, mix_prob=RESIZEMIX_PROB)
train_loader = DataLoader(train_mix_dataset, shuffle=True, generator=loader_generator, **loader_options)
val_loader = DataLoader(val_subset, shuffle=False, **loader_options)
clean_train_loader = DataLoader(clean_train_subset, shuffle=False, **loader_options)

print('Training batches:', len(train_loader))
print('Validation batches:', len(val_loader))

In [ ]:
# Sample check: look at these before training. Top row plain, other rows are real training images, with the pasted label share.
show_mix_samples(train_dataset, train_subset, train_mix_dataset)

## 3. Model, optimizer and cosine scheduler

In [ ]:
# PCCNN (src/model/pc_cnn_ep.py): StandardCNN layers + centered EP learning
model = PCCNN(num_classes=NUM_CLASSES, block3_dropout=0.15, block4_dropout=0.25, classifier_dropout=0.35,
              level4_dilation=2, beta=EP_BETA, steps=EP_STEPS, label_smoothing=LABEL_SMOOTHING, seed=SEED).to(DEVICE)

total_params = count_parameters(model)
print(f"Parameters: {total_params:,} ({total_params / 1e6:.3f}M)")
print(f"Architecture: 8 conv (2+2+2+2, conv8 dilation 2) | BatchNorm x9 (8 conv + dense) | LeakyReLU 0.05 | MaxPool x3 | Dropout2d 0.15, 0.25 | GAP 512 → dense 512 (dropout 0.35) → {NUM_CLASSES}")
print(f'Centered EP: K={model.steps} sweeps per phase, beta={model.beta}, dropout on states {sorted(model.dropout)} = {list(model.dropout.values())}')

# Loss, optimizer and cosine LR schedule (per step), same as BP
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.SGD(model.parameters(), lr=INITIAL_LR, momentum=0.9, weight_decay=WEIGHT_DECAY, nesterov=True)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS * len(train_loader), eta_min=MIN_LR)

In [ ]:
# Timing check (optional): BP vs EP step time and GPU memory, on copies of the model (training is not affected)
step_times = compare_step_time(model, BATCH_SIZE, NUM_CLASSES, LABEL_SMOOTHING, DEVICE, len(train_loader), EPOCHS)

## 4. Checkpoints and resume

In [ ]:
# Run config: saved in latest_checkpoint.pth; RESUME refuses to continue if it differs.
CONFIG = dict(num_classes=NUM_CLASSES, batch_size=BATCH_SIZE, epochs=EPOCHS, train_per_class=TRAIN_PER_CLASS,
              val_per_class=VAL_PER_CLASS, classifier='8conv_v2_gap_dense_fc', classifier_dropout=CLASSIFIER_DROPOUT,
              augmentation='resizemix_half_of_images_randaugment2m5_after', resizemix_scale=RESIZEMIX_SCALE,
              resizemix_prob=RESIZEMIX_PROB,
              method='centered_ep', ep_steps=EP_STEPS, ep_beta=EP_BETA)

# Best / latest checkpoints, random states and resume: src/utils/checkpoint.py
ckpt = CheckpointManager(SAVE_DIR, model, optimizer, scheduler, CONFIG, DEVICE, generators={'loader': loader_generator, 'ep': model.dropout_generator})
history, best_acc, best_epoch, start_epoch = ckpt.start(RESUME)

## 5. Training and validation

In [ ]:
# Training loop: TrainModel (src/utils/train_model.py); PCCNN trains with centered EP by default: each batch's gradient comes from centered EP (model.ep_gradients) instead of backprop.
# Same loop for BP and EP; writes metrics.csv at the end and records the time per epoch.
trainer = TrainModel(model, optimizer, scheduler, criterion, DEVICE)
best_acc, best_epoch = trainer.fit(train_loader, val_loader, clean_train_loader, EPOCHS, history, SAVE_DIR,
                                   ckpt.save_latest, start_epoch=start_epoch, best_acc=best_acc, best_epoch=best_epoch,
                                   clean_eval_interval=CLEAN_EVAL_INTERVAL,
                                   best_info={'class_names': CLASS_NAMES,
                                              'experiment': '39 PCCNN, centered EP (K=10, beta=0.01), ResizeMix on half of images, RandAugment after mixing'})

In [ ]:
# All metrics per epoch (from metrics.csv), then best and mean val of the last 5 epochs
print_metrics(SAVE_DIR / 'metrics.csv')

## 6. Training and validation curves

In [ ]:
plot_training_curves(history, train_label='Train (EP, 50% ResizeMix)')

ckpt.load_best()
model.eval()

clean_train_loss, clean_train_acc = trainer.run_epoch(clean_train_loader)
clean_val_loss, clean_val_acc = trainer.run_epoch(val_loader)

print(
    f'Best checkpoint: clean train {clean_train_acc:.2f}%, clean val {clean_val_acc:.2f}%, gap {clean_train_acc - clean_val_acc:.2f} pp')